# Decode APRS

This notebook walks through the APRS receive chain using local IQ samples if available, or a synthetic Bell 202 AFSK-over-FM fallback if not. The goal is to make the FM -> AFSK -> symbol decision path visible.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from rf_utils import *
from IPython.display import Audio, Markdown, display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

%matplotlib widget

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})


In [ ]:
CAPTURE_ROOT = ROOT / "assets" / "local"
capture_status = probe_rtlsdr()
display(Markdown(
    f"**RTL-SDR status:** installed={capture_status['installed']}, "
    f"available={capture_status['available']}. {capture_status['message']}"
))


## Capture Source and Synthetic Frame

Local path: `assets/local/aprs_afsk_iq.npz`. Synthetic fallback encodes a simple alternating Bell 202 tone sequence that stands in for an AX.25 frame.

In [ ]:
capture_path = CAPTURE_ROOT / "aprs_afsk_iq.npz"
symbol_rate = 1200
mark = 1200
space = 2200

if capture_path.exists():
    fs_iq, iq = load_complex_capture(capture_path)
    print(f"Loaded local capture: {capture_path.name}, fs={fs_iq}")
else:
    fs_iq = 96_000
    bits = np.array([0, 1, 1, 0, 1, 0, 0, 1] * 30)
    samples_per_symbol = fs_iq // symbol_rate
    t_sym = np.arange(samples_per_symbol) / fs_iq
    afsk = np.concatenate([
        np.cos(2 * np.pi * (mark if bit else space) * t_sym) for bit in bits
    ])
    iq = synthesize_fm_iq(normalize(afsk), fs=fs_iq, carrier_offset=18_000, freq_dev=3000)
    print("Using synthetic APRS-like Bell 202 fallback.")


In [ ]:
audio = fm_demodulate_iq(complex_mix_down(iq, fs_iq, 18_000), fs=fs_iq, audio_cutoff=3500)
f_mark, p_mark = power_spectrum(audio, fs_iq, nfft=8192)

samples_per_symbol = fs_iq // symbol_rate
usable = len(audio) // samples_per_symbol
audio_symbols = audio[: usable * samples_per_symbol].reshape(usable, samples_per_symbol)
t_sym = np.arange(samples_per_symbol) / fs_iq
mark_ref = np.cos(2 * np.pi * mark * t_sym)
space_ref = np.cos(2 * np.pi * space * t_sym)
mark_energy = np.abs(audio_symbols @ mark_ref)
space_energy = np.abs(audio_symbols @ space_ref)
decoded_bits = (mark_energy > space_energy).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
plot_waveform(audio[:12000], fs=fs_iq, ax=axes[0], title="Recovered AFSK audio")
plot_spectrum(audio, fs=fs_iq, ax=axes[1], title="Bell 202 tones")
axes[1].set_xlim(0, 3000)
axes[1].set_ylim(-100, 5)
axes[2].plot(decoded_bits[:80], drawstyle="steps-post")
axes[2].set_title("Detected bits")
axes[2].set_xlabel("Symbol index")
axes[2].set_ylim(-0.2, 1.2)
plt.tight_layout()

display(Markdown(f"**First 64 decoded bits:** `{''.join(map(str, decoded_bits[:64]))}`"))
display(audio_player(resample_signal(audio, fs_iq, 44_100), rate=44_100))


## Key Takeaway

APRS decoding is a chain of simpler problems. First recover audio, then separate mark and space energy, then turn symbol decisions into bits and frames.